# Lakehouse Platform - PySpark + Iceberg Intro

This notebook demonstrates:
- Connecting PySpark to MinIO (S3 Compatible)
- Creating Apache Iceberg tables
- Writing and reading data
- Time Travel with Iceberg


In [9]:
from lakehouse_utils import get_spark, get_s3_client
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = get_spark()
print(f'Spark version: {spark.version}')
spark

Spark version: 3.5.0


## 1. Create a sample DataFrame

In [10]:
from datetime import date

data = [
    ('P001', date(2024, 1, 1), 150, 4500.00, 10),
    ('P002', date(2024, 1, 1), 80,  2400.00, 5),
    ('P003', date(2024, 1, 2), 200, 6000.00, 12),
    ('P001', date(2024, 1, 2), 120, 3600.00, 8),
]

schema = StructType([
    StructField('product_id',   StringType(),  False),
    StructField('order_date',   DateType(),    False),
    StructField('units_sold',   IntegerType(), True),
    StructField('gross_revenue',DoubleType(),  True),
    StructField('unique_customers', IntegerType(), True),
])

df = spark.createDataFrame(data, schema)
df.show()

+----------+----------+----------+-------------+----------------+
|product_id|order_date|units_sold|gross_revenue|unique_customers|
+----------+----------+----------+-------------+----------------+
|      P001|2024-01-01|       150|       4500.0|              10|
|      P002|2024-01-01|        80|       2400.0|               5|
|      P003|2024-01-02|       200|       6000.0|              12|
|      P001|2024-01-02|       120|       3600.0|               8|
+----------+----------+----------+-------------+----------------+



## 2. Write to Iceberg Table

In [11]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS lakehouse.demo')

df.writeTo('lakehouse.demo.product_metrics') \
  .partitionedBy('order_date') \
  .createOrReplace()

print('Table created successfully!')

Table created successfully!


## 3. Query with Spark SQL

In [12]:
spark.sql("""
    SELECT 
        product_id,
        SUM(gross_revenue) AS total_revenue,
        SUM(units_sold)    AS total_units
    FROM lakehouse.demo.product_metrics
    GROUP BY product_id
    ORDER BY total_revenue DESC
""").show()

+----------+-------------+-----------+
|product_id|total_revenue|total_units|
+----------+-------------+-----------+
|      P001|       8100.0|        270|
|      P003|       6000.0|        200|
|      P002|       2400.0|         80|
+----------+-------------+-----------+



## 4. Iceberg Time Travel - Table History

In [13]:
spark.sql('SELECT * FROM lakehouse.demo.product_metrics.history').show(truncate=False)

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-05-17 02:26:07.782|7223063300578988143|NULL     |false              |
|2026-05-17 02:27:00.627|8167015808619192081|NULL     |false              |
|2026-05-17 02:27:52.418|107423599264900844 |NULL     |false              |
|2026-05-17 02:42:38.631|9186708736677313753|NULL     |false              |
|2026-05-17 13:59:40.588|8087794346564669253|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+



## 5. List MinIO buckets with boto3

In [14]:
s3 = get_s3_client()
buckets = s3.list_buckets()['Buckets']
print('Available buckets:')
for b in buckets:
    print(f"  - {b['Name']}  (created: {b['CreationDate']})")


Available buckets:
  - bronze  (created: 2026-05-17 00:55:28.769000+00:00)
  - checkpoints  (created: 2026-05-17 00:55:28.994000+00:00)
  - gold  (created: 2026-05-17 00:55:28.918000+00:00)
  - logs  (created: 2026-05-17 00:55:29.067000+00:00)
  - raw  (created: 2026-05-17 00:55:28.694000+00:00)
  - silver  (created: 2026-05-17 00:55:28.844000+00:00)
  - temp  (created: 2026-05-17 00:55:29.143000+00:00)
